logging

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)
from datetime import datetime, timezone
import uuid

PIPELINE_NAME = "ecommerce_lakehouse_pipeline"
TASK_NAME = "silver_transformation"
AUDIT_TABLE = "ecommerce.gold.pipeline_audit"

# Use a unique ID for each notebook execution.
execution_id = str(uuid.uuid4())

# try:
#     job_run_id = spark.conf.get(
#         "spark.databricks.job.runId",
#         "manual-or-unavailable"
#     )
# except Exception:
#     job_run_id = "manual-or-unavailable"

try:
    job_run_id = dbutils.widgets.get("job_run_id")
except Exception:
    job_run_id = "manual-or-unavailable"

task_started_at = datetime.now(timezone.utc).replace(tzinfo=None)


def write_audit(status, records_processed=None, details=None):
    task_ended_at = datetime.now(timezone.utc).replace(tzinfo=None)

    schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_status", StringType(), False),
        StructField("records_processed", LongType(), True),
        StructField("run_started_at", TimestampType(), False),
        StructField("run_ended_at", TimestampType(), False),
        StructField("details", StringType(), True),
        StructField("job_run_id", StringType(), True),
        StructField("execution_id", StringType(), False)
    ])

    record = [(
        str(PIPELINE_NAME),
        str(TASK_NAME),
        str(status),
        int(records_processed) if records_processed is not None else None,
        task_started_at,
        task_ended_at,
        str(details) if details is not None else None,
        str(job_run_id),
        str(execution_id)
    )]

    audit_df = spark.createDataFrame(record, schema=schema)

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE)
    )



write_audit("RUNNING", details="Silver transformation started")
print("Silver audit started:", execution_id)

Read Bronze tables, clean records, convert types, deduplicate, and write Silver tables.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_orders = spark.table("ecommerce.bronze.orders")

print("Bronze count:", bronze_orders.count())

display(bronze_orders.limit(10))


In [0]:

silver_orders_stage = (
    bronze_orders
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )
    .withColumn(
        "order_purchase_timestamp",
        F.to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "order_approved_at",
        F.to_timestamp("order_approved_at")
    )
    .withColumn(
        "order_delivered_carrier_date",
        F.to_timestamp("order_delivered_carrier_date")
    )
    .withColumn(
        "order_delivered_customer_date",
        F.to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "order_estimated_delivery_date",
        F.to_timestamp("order_estimated_delivery_date")
    )
)

silver_orders_stage.printSchema()
display(silver_orders_stage.limit(10))


Validation

In [0]:

invalid_orders = silver_orders_stage.filter(
    F.col("order_id").isNull()
    | (F.trim(F.col("order_id")) == "")
    | F.col("customer_id").isNull()
    | (F.trim(F.col("customer_id")) == "")
    | F.col("order_status").isNull()
    | (F.trim(F.col("order_status")) == "")
    | F.col("order_purchase_timestamp").isNull()
)

valid_orders = silver_orders_stage.filter(
    F.col("order_id").isNotNull()
    & (F.trim(F.col("order_id")) != "")
    & F.col("customer_id").isNotNull()
    & (F.trim(F.col("customer_id")) != "")
    & F.col("order_status").isNotNull()
    & (F.trim(F.col("order_status")) != "")
    & F.col("order_purchase_timestamp").isNotNull()
)

print("Invalid records:", invalid_orders.count())
print("Valid records:", valid_orders.count())


In [0]:

dedup_window = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("ingestion_timestamp").desc())
)

deduplicated_orders = (
    valid_orders
    .withColumn(
        "row_num",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("Before deduplication:", valid_orders.count())
print("After deduplication:", deduplicated_orders.count())


In [0]:

(
    deduplicated_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.silver.orders")
)

print("Silver orders table created successfully!")


In [0]:
%sql

SELECT
    order_status,
    COUNT(*) AS total_orders,
    MIN(order_purchase_timestamp) AS first_order,
    MAX(order_purchase_timestamp) AS latest_order
FROM ecommerce.silver.orders
GROUP BY order_status
ORDER BY total_orders DESC;


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_order_ids
FROM ecommerce.silver.orders;


Collecting data from other tables to prepare fact and dim tables

In [0]:
display(spark.table("ecommerce.bronze.order_items").printSchema())

In [0]:

from pyspark.sql import functions as F

# Read the already-persisted Bronze tables
order_items_bronze = spark.table("ecommerce.bronze.order_items")
payments_bronze = spark.table("ecommerce.bronze.payments")

# Clean and explicitly type order items
silver_order_items = (
    order_items_bronze
    .withColumn("order_id", F.col("order_id").cast("string"))
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("product_id", F.col("product_id").cast("string"))
    .withColumn("seller_id", F.col("seller_id").cast("string"))
    .withColumn(
        "shipping_limit_date",
        F.to_timestamp("shipping_limit_date")
    )
    .withColumn("price", F.col("price").cast("decimal(12,2)"))
    .withColumn(
        "freight_value",
        F.col("freight_value").cast("decimal(12,2)")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("order_item_id").isNotNull()
        & F.col("price").isNotNull()
        & F.col("freight_value").isNotNull()
    )
)

# Clean and explicitly type payments
silver_payments = (
    payments_bronze
    .withColumn("order_id", F.col("order_id").cast("string"))
    .withColumn(
        "payment_sequential",
        F.col("payment_sequential").cast("int")
    )
    .withColumn(
        "payment_installments",
        F.col("payment_installments").cast("int")
    )
    .withColumn(
        "payment_value",
        F.col("payment_value").cast("decimal(12,2)")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("payment_value").isNotNull()
    )
)

print("Silver order items schema:")
silver_order_items.printSchema()

print("Silver payments schema:")
silver_payments.printSchema()

print("Order item rows:", silver_order_items.count())
print("Payment rows:", silver_payments.count())

# These are derived Silver tables, so recreate them
# to remove any conflicting legacy schemas.
spark.sql("DROP TABLE IF EXISTS ecommerce.silver.order_items")
spark.sql("DROP TABLE IF EXISTS ecommerce.silver.payments")

(
    silver_order_items.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce.silver.order_items")
)

(
    silver_payments.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce.silver.payments")
)

print("Silver order_items and payments created successfully.")

In [0]:
display(spark.table("ecommerce.bronze.order_items").printSchema())

In [0]:

items_per_order = (
    spark.table("ecommerce.silver.order_items")
    .groupBy("order_id")
    .agg(
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_revenue"),
        F.count("*").alias("item_count")
    )
)

payments_per_order = (
    spark.table("ecommerce.silver.payments")
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_total"),
        F.count("*").alias("payment_count")
    )
)

display(items_per_order.limit(10))
display(payments_per_order.limit(10))


In [0]:
silver_order_count = spark.table("ecommerce.silver.orders").count()
silver_item_count = spark.table("ecommerce.silver.order_items").count()
silver_payment_count = spark.table("ecommerce.silver.payments").count()

write_audit(
    "SUCCESS",
    records_processed=silver_order_count,
    details=(
        f"Silver transformation completed. "
        f"Orders={silver_order_count}, "
        f"Order items={silver_item_count}, "
        f"Payments={silver_payment_count}"
    )
)

print("Silver audit completed successfully.")

In [0]:
%sql
SELECT *
FROM ecommerce.gold.pipeline_audit
WHERE task_name = 'silver_transformation'
ORDER BY run_started_at DESC;